In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [62]:
df=pd.read_csv("../../Datasets/cardekho_imputated.csv")

In [63]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


DATA CLEANING 

In [64]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [65]:
#remove unecessary columns
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace=True)
df.head()

,Unnamed: 0,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [66]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [67]:
## getting all different features
num_features=[feature for feature in df.columns if df[feature].dtype !='O']
cat_features=[feature for feature in df.columns if df[feature].dtype == 'O']
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25 ]
continous_featurs=[feature for feature in num_features if feature not in discrete_features ]

In [68]:
#independent and dependent
x=df.drop(['selling_price'],axis=1)
y=df['selling_price']

In [69]:
x.head()

,Unnamed: 0,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [70]:
#feature encoding adn scaling
df['model'].unique

<bound method Series.unique of 0            Alto
1           Grand
2             i20
3            Alto
4        Ecosport
           ...   
15406         i10
15407      Ertiga
15408       Rapid
15409      XUV500
15410        City
Name: model, Length: 15411, dtype: object>

In [71]:
df['model'].value_counts()

model
i20            906
Swift Dzire    890
Swift          781
Alto           778
City           757
              ... 
Ghibli           1
Altroz           1
GTC4Lusso        1
Aura             1
Gurkha           1
Name: count, Length: 120, dtype: int64

In [72]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
x['model']=le.fit_transform(x['model'])

In [73]:
#create column transformer with 3 types of transformations 
num_features=x.select_dtypes(exclude="object").columns
onehot_columns=['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder(drop="first")
preprocessor=ColumnTransformer(
    [
        ('OneHotEncoder',oh_transformer,onehot_columns),
        ('StandardScaler',numeric_transformer,num_features)
    ],remainder='passthrough'
)



In [74]:
x=preprocessor.fit_transform(x)

In [75]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=43)

Model training and nmodel selection

In [76]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [81]:
#create a function to Evaluate Model
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(true,predicted)
    return mae,rmse,r2_square

In [82]:
##Beginning model training
models={
    "Random Forest":RandomForestRegressor(),
    "Linear Regression":LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "K-Neighbors Regressor":KNeighborsRegressor(),
    "Decision Tree":DecisionTreeRegressor(),
    "Random Forest Regressor":RandomForestRegressor()
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)

    #make predictions
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)

    #training set performance
    model_train_mae,model_train_rmse,model_train_r2_score=evaluate_model(y_train,y_train_pred)

    #test set performance
    model_test_mae,model_test_rmse,model_test_r2_score=evaluate_model(y_test,y_test_pred)


    print(list(models.keys())[i])

    print('Model performance for Training set')
    print("-Root Mean Squared Error:{:.4f}".format(model_train_rmse))
    print("-Mean Absolute Error:{:.4f}".format(model_train_mae))
    print("-R2 Score:{:.4f}".format(model_train_r2_score))
    print('----------------------------------')
    
    print('Model performance for Test set')
    print("-Root Mean Squared Error:{:.4f}".format(model_test_rmse))
    print("-Mean Absolute Error:{:.4f}".format(model_test_mae))
    print("-R2 Score:{:.4f}".format(model_test_r2_score))

    print('=====================================')

Random Forest
Model performance for Training set
-Root Mean Squared Error:120374.3807
-Mean Absolute Error:36239.8169
-R2 Score:0.9813
----------------------------------
Model performance for Test set
-Root Mean Squared Error:289102.6479
-Mean Absolute Error:99440.7102
-R2 Score:0.9019
Linear Regression
Model performance for Training set
-Root Mean Squared Error:543170.0187
-Mean Absolute Error:264333.2077
-R2 Score:0.6202
----------------------------------
Model performance for Test set
-Root Mean Squared Error:548922.7029
-Mean Absolute Error:268377.8047
-R2 Score:0.6464
Lasso
Model performance for Training set
-Root Mean Squared Error:543170.0228
-Mean Absolute Error:264331.7307
-R2 Score:0.6202
----------------------------------
Model performance for Test set
-Root Mean Squared Error:548923.7367
-Mean Absolute Error:268377.3329
-R2 Score:0.6464
Ridge
Model performance for Training set
-Root Mean Squared Error:543170.3348
-Mean Absolute Error:264294.0303
-R2 Score:0.6202
-----------

In [83]:
##hyper parameter tunning 
knn_params={"n_neighbors":[2,3,10,20,40,50]}
rf_params={"max_depth":[5,8,15,None,10],
           "max_features":[5,7,"auto",8],
           "min_samples_split":[2,8,15,20],
           "n_estimators":[100,200,500,1000]
          }

In [85]:
#models list for Hyperparameter tunning
randomcv_models=[('KNN',KNeighborsRegressor(),knn_params),
                ("RF",RandomForestRegressor(),rf_params)]

In [86]:
from sklearn.model_selection import RandomizedSearchCV
model_param={}
for name,model,params in randomcv_models:
    random=RandomizedSearchCV(estimator=model,
                           param_distributions=params,
                           n_iter=100, 
                           cv=3,
                           verbose=2,
                           n_jobs=-1)
    random.fit(x_train,y_train)
    model_param[name]=random.best_params_
for model_name in model_param:
    print(f"---------------Best Params for {model_name}----------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
---------------Best Params for KNN----------------------
{'n_neighbors': 2}
---------------Best Params for RF----------------------
{'n_estimators': 1000, 'min_samples_split': 2, 'max_features': 8, 'max_depth': None}


In [92]:
##Retraining the models with best parameters 
models={
    "Random Forest":RandomForestRegressor(n_estimators=100,min_samples_split=2,max_features=8,max_depth=None,n_jobs=-1),
    "K-Neighbors Regressor":KNeighborsRegressor(n_neighbors=10,n_jobs=-1)

}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)

    #make predictions
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)

   #training set performance
    model_train_mae,model_train_rmse,model_train_r2_score=evaluate_model(y_train,y_train_pred)

    #test set performance
    model_test_mae,model_test_rmse,model_test_r2_score=evaluate_model(y_test,y_test_pred)


    print(list(models.keys())[i])

    print('Model performance for Training set')
    print("-Root Mean Squared Error:{:.4f}".format(model_train_rmse))
    print("-Mean Absolute Error:{:.4f}".format(model_train_mae))
    print("-R2 Score:{:.4f}".format(model_train_r2_score))
    print('----------------------------------')
    
    print('Model performance for Test set')
    print("-Root Mean Squared Error:{:.4f}".format(model_test_rmse))
    print("-Mean Absolute Error:{:.4f}".format(model_test_mae))
    print("-R2 Score:{:.4f}".format(model_test_r2_score))

    print('=====================================')

Random Forest
Model performance for Training set
-Root Mean Squared Error:158508.0775
-Mean Absolute Error:36377.0516
-R2 Score:0.9677
----------------------------------
Model performance for Test set
-Root Mean Squared Error:296837.5718
-Mean Absolute Error:97670.4344
-R2 Score:0.8966
K-Neighbors Regressor
Model performance for Training set
-Root Mean Squared Error:391772.8994
-Mean Absolute Error:113607.1881
-R2 Score:0.8024
----------------------------------
Model performance for Test set
-Root Mean Squared Error:387476.8575
-Mean Absolute Error:128577.7357
-R2 Score:0.8238
